# CCR Portfolio Simulation with Hull-White and GBM Risk Factors

This notebook demonstrates a clean public workflow for counterparty credit risk exposure analysis. It uses synthetic inputs and self-contained code from the `src/` folder.

## 1. Load configuration and generate scenarios

The scenario set includes EUR rates, USD rates and EUR/USD FX. Brownian shocks are correlated through a correlation matrix.

In [ ]:
from src.config import load_config
from src.scenario_generation import generate_scenarios

config = load_config('../data/synthetic_parameters.json')
scenarios = generate_scenarios(config)

print(scenarios.time_grid[:5])
print(scenarios.eur_rate.shape, scenarios.usd_rate.shape, scenarios.eur_usd_fx.shape)

## 2. Build and value the portfolio

The portfolio contains two synthetic interest-rate swaps. USD MtM is converted into EUR using the simulated EUR/USD FX path.

In [ ]:
from src.portfolio_valuation import build_portfolio, value_trades, aggregate_netting_set, apply_simple_collateral

trades = build_portfolio(config)
trade_mtms = value_trades(trades, scenarios)
netting_set_mtm = aggregate_netting_set(trade_mtms)

exposure_after_collateral = apply_simple_collateral(
    netting_set_mtm,
    threshold=config['portfolio']['collateral_threshold'],
    minimum_transfer_amount=config['portfolio']['minimum_transfer_amount'],
)

print([trade.trade_id for trade in trades])
print(netting_set_mtm.shape)

## 3. Compute EE and PFE

Expected Exposure (EE) is the average positive exposure at each time. Potential Future Exposure (PFE) is computed as a high quantile of the positive exposure distribution.

In [ ]:
from src.exposure_analysis import exposure_summary_frame

exposure_df = exposure_summary_frame(
    scenarios.time_grid,
    netting_set_mtm,
    exposure_after_collateral,
    quantile=0.95,
)

exposure_df.head()

## 4. Plot exposure profile

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))
plt.plot(exposure_df['time'], exposure_df['ee_raw'], label='EE raw')
plt.plot(exposure_df['time'], exposure_df['pfe_95_raw'], label='PFE 95% raw')
plt.plot(exposure_df['time'], exposure_df['ee_after_collateral'], label='EE after collateral')
plt.plot(exposure_df['time'], exposure_df['pfe_95_after_collateral'], label='PFE 95% after collateral')
plt.title('Portfolio Exposure Profile')
plt.xlabel('Time in years')
plt.ylabel('Exposure in EUR')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()